# 04 · Пары: DPO, ORPO, SimPO, KTO

Пара здесь предельно чистая: тот же промпт, `chosen` — верная метка, `rejected` — неверная. Модель
Брэдли–Терри $P(y_w \succ y_l) = \sigma(r_w - r_l)$, у DPO $r = \beta \log \frac{\pi_\theta}{\pi_{\text{ref}}}$,
ORPO и SimPO обходятся без референса, KTO учится по одному ответу с меткой.

Стартуем от SFT: адаптер вливается в веса и становится референсом. Так метод на парах двигает
границу решения, а не учит формат с нуля.

In [ ]:
import sys
sys.path.insert(0, "../..")

from src import data, infer
from src import filter as F

from peft import LoraConfig, PeftModel
from trl import DPOConfig, DPOTrainer, KTOConfig, KTOTrainer
from trl.experimental.cpo import CPOConfig, CPOTrainer      # SimPO is CPO with loss_type="simpo"
from trl.experimental.orpo import ORPOConfig, ORPOTrainer

model, tokenizer = infer.load_model()
model = PeftModel.from_pretrained(model, F.RUNS / "sft-adapter").merge_and_unload()

train = F.load("train")
pairs = train.select_columns(["prompt", "chosen", "rejected"])
unpaired = data.to_kto(train)

In [ ]:
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    # attention and MLP projections of the language stack; the vision tower is excluded, there are no images
    target_modules=r"^(?!.*(visual|vision)).*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
    use_rslora=True,
    task_type="CAUSAL_LM",
)

COMMON = dict(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=5e-5,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=1024,
    logging_steps=10,
    save_strategy="no",
    report_to=[],
    seed=42,
)
METHODS = {
    "dpo":   (DPOConfig,  DPOTrainer,  {"beta": 0.1}),
    "orpo":  (ORPOConfig, ORPOTrainer, {"beta": 0.1}),
    "simpo": (CPOConfig,  CPOTrainer,  {"loss_type": "simpo", "cpo_alpha": 0.05, "simpo_gamma": 0.5, "learning_rate": 1e-5}),
    "kto":   (KTOConfig,  KTOTrainer,  {"beta": 0.1}),
}

for name, (Config, Trainer, specific) in METHODS.items():
    print("═" * 78, name.upper())
    config = Config(output_dir=str(F.RUNS / name), **{**COMMON, **specific})
    trainer = Trainer(model=model, args=config, train_dataset=unpaired if name == "kto" else pairs,
                            processing_class=tokenizer, peft_config=lora)
    history = trainer.train()
    tuned = trainer.model
    print(f"loss {history.training_loss:.3f} | {infer.free(trainer)}")
    del trainer
    F.evaluate(tuned, tokenizer, name, note=f"{name} on top of SFT, 2 epochs")
    tuned.save_pretrained(F.RUNS / f"{name}-adapter")
    model = tuned.unload()
    del tuned

F.show()